# Stage 3 — LLM Explanation Generation

**Reads:** `outputs/ontology_results.json`  
**Writes:** `outputs/explanations.json`

Generates a natural-language, user-adaptive explanation for each text using
the Qwen LLM, grounded in the ontology triples from Stage 2.

Each record saved = Stage 2 record + `"explanation": "..."`

> **Tip:** You can test a single explanation in Section 6 before running the
> full batch. The LLM is only loaded once.

## 1. Imports

In [1]:
from config import (
    EXPLANATIONS_PATH,
    ONTOLOGY_RESULTS_PATH,
    NUM_BEAMS,
    USE_CONSTRAINED_DECODING,
    LAMBDA_MAP,
    USER_CATEGORY,
)
from constrained_decoding import ReadabilityBeamGenerator
from model_loaders import load_llm
from pipeline_helpers import (
    SYSTEM_PROMPT,
    build_prompt,
    checkpoint_exists,
    generate_explanation,
    load_checkpoint,
    save_checkpoint,
)

## 2. Configuration

In [2]:
# Set True to re-run even if explanations.json already exists
FORCE_RERUN = True

## 3. Checkpoint check

In [3]:
if checkpoint_exists(EXPLANATIONS_PATH) and not FORCE_RERUN:
    print(f"⚠️  Checkpoint found at '{EXPLANATIONS_PATH}'.")
    print("    Set FORCE_RERUN = True to overwrite.")
    print("    Loading existing results …")
    results = load_checkpoint(EXPLANATIONS_PATH)
else:
    results = None
    print("No checkpoint found (or FORCE_RERUN=True). Will generate explanations.")

No checkpoint found (or FORCE_RERUN=True). Will generate explanations.


## 4. Load Stage 2 output

In [4]:
if results is None:
    if not ONTOLOGY_RESULTS_PATH.exists():
        raise FileNotFoundError(
            f"Ontology results not found at '{ONTOLOGY_RESULTS_PATH}'.\n"
            "Please run 02_ontology.ipynb first."
        )
    ontology_data = load_checkpoint(ONTOLOGY_RESULTS_PATH)
    print(f"Loaded {len(ontology_data)} texts from Stage 2.")

[Checkpoint] Loaded 50 records ← 'outputs\ontology_results_beginner.json'
Loaded 50 texts from Stage 2.


## 5. Load LLM

In [5]:
if results is None:
    llm_tokenizer, llm_model = load_llm()
    generator = None
    if USE_CONSTRAINED_DECODING:
        generator = ReadabilityBeamGenerator(
            model=llm_model,
            tokenizer=llm_tokenizer,
            num_beams=NUM_BEAMS,
        )
        print(
            f"Constrained decoding enabled | beams={NUM_BEAMS} | ",
            f"lambda(default={USER_CATEGORY})={LAMBDA_MAP.get(USER_CATEGORY, LAMBDA_MAP['EXPERT'])}",
        )
else:
    generator = None

[Loader] Loading LLM 'Qwen/Qwen2.5-1.5B-Instruct' …


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[Loader] LLM ready.



## 6. [Optional] Preview prompt for a single sample

Run this cell to inspect what the LLM will receive before running the full batch.

In [6]:
if results is None:
    sample = ontology_data[0]          # ← change index to preview a different text
    prompt = build_prompt(
        predicted_class=sample["predicted_class"],
        feature_data=sample["feature_data"],
        user_category=sample["user_category"],
    )
    print("── SYSTEM PROMPT ─────────────────────────────────────────")
    print(SYSTEM_PROMPT)
    print("\n── USER PROMPT ───────────────────────────────────────────")
    print(prompt)

── SYSTEM PROMPT ─────────────────────────────────────────
You are a biomedical explanation assistant. Your job is to generate clear, accurate natural language explanations of why a machine learning model made a specific biomedical prediction. You are given:
- The model's predicted class
- Key tokens identified by LIME (local feature attribution) as influential in the prediction
- Ontology-derived ancestor chains for each token, showing its place in the biomedical concept hierarchy

Your explanations must be grounded strictly in the provided features and ontology context. Do not introduce facts, diseases, or concepts not present in the input.

Adapt your explanation style based on the user category:
- BEGINNER: Use plain, everyday language. Avoid technical jargon. Explain medical terms when they appear. Keep sentences short. The goal is comprehension, not completeness.
- INTERMEDIATE: Balance accessibility with domain accuracy. Define specialized terms briefly. Use medical vocabulary w

## 7. Generate explanations (full batch)

In [7]:
if results is None:
    results = []
    for i, item in enumerate(ontology_data):
        print(f"\nGenerating explanation {i + 1}/{len(ontology_data)} …")
        print(f"  Class    : {item['predicted_class']}")
        print(f"  User     : {item['user_category']}")
        print(f"  Features : {[f['feature_word'] for f in item['feature_data']]}")
        if USE_CONSTRAINED_DECODING:
            lam = LAMBDA_MAP.get(item["user_category"], LAMBDA_MAP["EXPERT"])
            print(f"  Lambda   : {lam}")

        explanation = generate_explanation(
            predicted_class=item["predicted_class"],
            feature_data=item["feature_data"],
            user_category=item["user_category"],
            tokenizer=llm_tokenizer,
            model=llm_model,
            generator=generator,
        )

        print(f"  Output   : {explanation[:120]}…")
        results.append({**item, "explanation": explanation})

    print("\n✅ Explanation generation complete.")


Generating explanation 1/50 …
  Class    : Digestive system diseases
  User     : BEGINNER
  Features : ['liver']
  Output   : assistant
The model predicted that the patient has digestive system diseases because it found the presence of the liver …

Generating explanation 2/50 …
  Class    : Cardiovascular diseases
  User     : BEGINNER
  Features : []
  Output   : The model predicted Cardiovascular diseases, but no ontology-based features were available to explain this decision.…

Generating explanation 3/50 …
  Class    : Digestive system diseases
  User     : BEGINNER
  Features : ['ascites', 'peritoneum', 'Endometriosis']
  Output   : assistant
The model predicted that someone has a digestive system disease because it found three key features related to…

Generating explanation 4/50 …
  Class    : Cardiovascular diseases
  User     : BEGINNER
  Features : ['claudication', 'pain']
  Output   : assistant
The model predicted cardiovascular diseases because it detected symptoms like c

## 8. Inspect explanations

In [8]:
for i, r in enumerate(results):
    print(f"\n── Text {i + 1} ──────────────────────────")
    print(f"Class   : {r['predicted_class']}")
    print(f"User    : {r['user_category']}")
    print(f"\nExplanation:\n{r['explanation']}")
    print()


── Text 1 ──────────────────────────
Class   : Digestive system diseases
User    : BEGINNER

Explanation:
assistant
The model predicted that the patient has digestive system diseases because it found the presence of the liver in the data. In the biomedical ontology, the liver is an organ located in the abdomen, which belongs to the digestive system. Therefore, if the model detected the liver in the dataset, it likely inferred that the patient might have conditions related to the digestive tract, such as gastritis, hepatitis, or other issues affecting this part of the body.


── Text 2 ──────────────────────────
Class   : Cardiovascular diseases
User    : BEGINNER

Explanation:
The model predicted Cardiovascular diseases, but no ontology-based features were available to explain this decision.


── Text 3 ──────────────────────────
Class   : Digestive system diseases
User    : BEGINNER

Explanation:
assistant
The model predicted that someone has a digestive system disease because it fou

## 9. Save checkpoint

In [9]:
save_checkpoint(results, EXPLANATIONS_PATH)
print(f"\n➡️  Continue to notebook 04_analysis.ipynb")

[Checkpoint] Saved 50 records → 'outputs\explanations_beginner.json'

➡️  Continue to notebook 04_analysis.ipynb
